# Dynamic Pricing Optimization - Data Exploration and Preprocessing
## Phase 1: Exploratory Data Analysis (EDA)

**Project Goal:** Build a data-driven dynamic pricing system that predicts demand and optimizes revenue

**This Notebook Covers:**
1. Data Loading and Initial Inspection
2. Data Quality Assessment
3. Statistical Summary and Distribution Analysis
4. Missing Values and Outliers Detection
5. Temporal Trends Analysis
6. Feature Correlation Analysis
7. Data Cleaning and Preprocessing
8. Save Cleaned Data for Modeling

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Data Loading and Initial Inspection

In [ ]:
# Load the Walmart Sales dataset
df = pd.read_csv('../data/Walmart.csv')

print("Dataset loaded successfully!")
print(f"\nDataset Shape: {df.shape}")
print(f"Number of Records: {df.shape[0]:,}")
print(f"Number of Features: {df.shape[1]}")

In [ ]:
# Display first few rows
print("\n=== First 10 Records ===")
df.head(10)

In [ ]:
# Display last few rows
print("\n=== Last 5 Records ===")
df.tail()

In [ ]:
# Dataset information
print("\n=== Dataset Information ===")
df.info()

In [ ]:
# Display column names and data types
print("\n=== Column Details ===")
for col in df.columns:
    print(f"{col:20} | {str(df[col].dtype):15} | Non-null: {df[col].count():6}")

## 2. Data Quality Assessment

In [ ]:
# Check for missing values
print("\n=== Missing Values Analysis ===")
missing_data = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_data) == 0:
    print("✓ No missing values found in the dataset!")
else:
    print(missing_data)

In [ ]:
# Check for duplicate rows
print("\n=== Duplicate Records Check ===")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    print(f"Percentage of duplicates: {(duplicates/len(df))*100:.2f}%")
else:
    print("✓ No duplicate records found!")

In [ ]:
# Unique values in categorical/discrete columns
print("\n=== Unique Values Analysis ===")
print(f"Number of unique stores: {df['Store'].nunique()}")
print(f"Store IDs range: {df['Store'].min()} to {df['Store'].max()}")
print(f"\nNumber of unique dates: {df['Date'].nunique()}")
print(f"\nHoliday Flag values: {df['Holiday_Flag'].unique()}")
print(f"Holiday weeks count: {df['Holiday_Flag'].sum()}")
print(f"Non-holiday weeks count: {len(df) - df['Holiday_Flag'].sum()}")

## 3. Statistical Summary and Distribution Analysis

In [ ]:
# Statistical summary of numerical features
print("\n=== Statistical Summary ===")
df.describe().T

In [ ]:
# Detailed statistics for Weekly Sales (our target variable)
print("\n=== Weekly Sales Statistics ===")
print(f"Mean Weekly Sales: ${df['Weekly_Sales'].mean():,.2f}")
print(f"Median Weekly Sales: ${df['Weekly_Sales'].median():,.2f}")
print(f"Std Dev: ${df['Weekly_Sales'].std():,.2f}")
print(f"Min Weekly Sales: ${df['Weekly_Sales'].min():,.2f}")
print(f"Max Weekly Sales: ${df['Weekly_Sales'].max():,.2f}")
print(f"Range: ${df['Weekly_Sales'].max() - df['Weekly_Sales'].min():,.2f}")
print(f"\nSkewness: {df['Weekly_Sales'].skew():.3f}")
print(f"Kurtosis: {df['Weekly_Sales'].kurtosis():.3f}")

In [ ]:
# Distribution plots for numerical features
numerical_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

# Remove the extra subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.savefig('../output/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Distribution plots saved to output/feature_distributions.png")

In [ ]:
# Box plots to identify outliers
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    axes[idx].boxplot(df[col], vert=True)
    axes[idx].set_title(f'Box Plot: {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(axis='y', alpha=0.3)

fig.delaxes(axes[5])
plt.tight_layout()
plt.savefig('../output/boxplots_outliers.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Box plots saved to output/boxplots_outliers.png")

## 4. Outlier Detection and Analysis

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("\n=== Outlier Analysis (IQR Method) ===")
for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"\n{col}:")
    print(f"  Lower Bound: {lower:.2f}")
    print(f"  Upper Bound: {upper:.2f}")
    print(f"  Number of Outliers: {len(outliers)} ({(len(outliers)/len(df)*100):.2f}%)")

## 5. Temporal Trends Analysis

In [ ]:
# Convert Date column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Extract temporal features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter

print("✓ Temporal features extracted successfully!")
print(f"\nDate Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Time Period: {(df['Date'].max() - df['Date'].min()).days} days")
print(f"Years covered: {df['Year'].unique()}")

In [ ]:
# Weekly sales trend over time
plt.figure(figsize=(16, 6))
weekly_avg = df.groupby('Date')['Weekly_Sales'].mean().reset_index()
plt.plot(weekly_avg['Date'], weekly_avg['Weekly_Sales'], linewidth=2, color='blue')
plt.title('Average Weekly Sales Trend Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Average Weekly Sales ($)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../output/sales_trend_over_time.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Sales trend plot saved to output/sales_trend_over_time.png")

In [ ]:
# Monthly sales pattern
monthly_sales = df.groupby('Month')['Weekly_Sales'].agg(['mean', 'median', 'std']).reset_index()
monthly_sales.columns = ['Month', 'Mean_Sales', 'Median_Sales', 'Std_Sales']

fig, ax = plt.subplots(figsize=(14, 6))
x = monthly_sales['Month']
ax.bar(x, monthly_sales['Mean_Sales'], alpha=0.7, color='steelblue', label='Mean Sales')
ax.plot(x, monthly_sales['Median_Sales'], color='red', marker='o', linewidth=2, label='Median Sales')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Weekly Sales ($)', fontsize=12)
ax.set_title('Monthly Sales Patterns', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../output/monthly_sales_pattern.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n=== Monthly Sales Statistics ===")
print(monthly_sales)

In [ ]:
# Holiday vs Non-Holiday Sales Comparison
holiday_comparison = df.groupby('Holiday_Flag')['Weekly_Sales'].agg(['mean', 'median', 'count']).reset_index()
holiday_comparison['Holiday_Flag'] = holiday_comparison['Holiday_Flag'].map({0: 'Non-Holiday', 1: 'Holiday'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(holiday_comparison['Holiday_Flag'], holiday_comparison['mean'], color=['lightcoral', 'lightgreen'])
axes[0].set_title('Average Sales: Holiday vs Non-Holiday', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Weekly Sales ($)')
axes[0].grid(axis='y', alpha=0.3)

# Box plot
df['Holiday_Type'] = df['Holiday_Flag'].map({0: 'Non-Holiday', 1: 'Holiday'})
df.boxplot(column='Weekly_Sales', by='Holiday_Type', ax=axes[1])
axes[1].set_title('Sales Distribution: Holiday vs Non-Holiday', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Type')
axes[1].set_ylabel('Weekly Sales ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../output/holiday_sales_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Holiday vs Non-Holiday Sales ===")
print(holiday_comparison)

## 6. Feature Correlation Analysis

In [ ]:
# Correlation matrix
correlation_cols = ['Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
correlation_matrix = df[correlation_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../output/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Correlation with Weekly Sales ===")
sales_corr = correlation_matrix['Weekly_Sales'].sort_values(ascending=False)
print(sales_corr)

In [ ]:
# Scatter plots: Weekly Sales vs Economic Indicators
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

scatter_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
for idx, feature in enumerate(scatter_features):
    axes[idx].scatter(df[feature], df['Weekly_Sales'], alpha=0.3, s=20)
    axes[idx].set_xlabel(feature, fontsize=11)
    axes[idx].set_ylabel('Weekly Sales', fontsize=11)
    axes[idx].set_title(f'Sales vs {feature}', fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)
    
    # Add trend line
    z = np.polyfit(df[feature].dropna(), df['Weekly_Sales'][df[feature].notna()], 1)
    p = np.poly1d(z)
    axes[idx].plot(df[feature], p(df[feature]), "r--", alpha=0.8, linewidth=2)

plt.tight_layout()
plt.savefig('../output/scatter_plots_sales_vs_features.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Store-Level Analysis

In [ ]:
# Sales statistics by store
store_stats = df.groupby('Store')['Weekly_Sales'].agg([
    ('Mean_Sales', 'mean'),
    ('Median_Sales', 'median'),
    ('Total_Sales', 'sum'),
    ('Min_Sales', 'min'),
    ('Max_Sales', 'max'),
    ('Std_Sales', 'std')
]).reset_index()

print("\n=== Top 10 Stores by Average Weekly Sales ===")
print(store_stats.nlargest(10, 'Mean_Sales')[['Store', 'Mean_Sales', 'Total_Sales']])

print("\n=== Bottom 10 Stores by Average Weekly Sales ===")
print(store_stats.nsmallest(10, 'Mean_Sales')[['Store', 'Mean_Sales', 'Total_Sales']])

In [ ]:
# Visualize top 10 performing stores
top_10_stores = store_stats.nlargest(10, 'Mean_Sales')

plt.figure(figsize=(12, 6))
plt.barh(top_10_stores['Store'].astype(str), top_10_stores['Mean_Sales'], color='teal')
plt.xlabel('Average Weekly Sales ($)', fontsize=12)
plt.ylabel('Store ID', fontsize=12)
plt.title('Top 10 Stores by Average Weekly Sales', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../output/top_10_stores.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Data Preprocessing and Feature Engineering

In [ ]:
# Create a copy for preprocessing
df_clean = df.copy()

# Sort by Store and Date
df_clean = df_clean.sort_values(['Store', 'Date']).reset_index(drop=True)

print("✓ Data sorted by Store and Date")
print(f"\nCleaned dataset shape: {df_clean.shape}")

In [ ]:
# Add season feature
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df_clean['Season'] = df_clean['Month'].apply(get_season)
print("✓ Season feature added")

# Season distribution
print("\n=== Season Distribution ===")
print(df_clean['Season'].value_counts())

In [ ]:
# Add lag features (previous week's sales)
df_clean['Sales_Lag_1'] = df_clean.groupby('Store')['Weekly_Sales'].shift(1)
df_clean['Sales_Lag_2'] = df_clean.groupby('Store')['Weekly_Sales'].shift(2)

print("✓ Lag features created")
print(f"Missing values in Sales_Lag_1: {df_clean['Sales_Lag_1'].isnull().sum()}")
print(f"Missing values in Sales_Lag_2: {df_clean['Sales_Lag_2'].isnull().sum()}")

In [ ]:
# Add rolling statistics (moving averages)
df_clean['Sales_Rolling_Mean_4'] = df_clean.groupby('Store')['Weekly_Sales'].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean()
)
df_clean['Sales_Rolling_Std_4'] = df_clean.groupby('Store')['Weekly_Sales'].transform(
    lambda x: x.rolling(window=4, min_periods=1).std()
)

print("✓ Rolling statistics features added")

## 9. Price Simulation (Required for Dynamic Pricing)

In [ ]:
# Simulate price feature based on sales and demand patterns
# We'll create a realistic price that inversely correlates with sales volume

np.random.seed(42)

# Base price per store (varying by store performance)
store_base_price = df_clean.groupby('Store')['Weekly_Sales'].mean().rank(pct=True) * 20 + 80
df_clean['Base_Price'] = df_clean['Store'].map(store_base_price)

# Add seasonal variation to price
season_price_factor = {'Winter': 1.05, 'Spring': 1.0, 'Summer': 0.95, 'Fall': 1.02}
df_clean['Season_Price_Factor'] = df_clean['Season'].map(season_price_factor)

# Final simulated price with some random noise
df_clean['Price'] = (
    df_clean['Base_Price'] * 
    df_clean['Season_Price_Factor'] * 
    (1 + np.random.normal(0, 0.05, len(df_clean)))  # 5% random variation
).round(2)

# Ensure price is positive
df_clean['Price'] = df_clean['Price'].clip(lower=50)

print("✓ Price feature simulated successfully!")
print(f"\nPrice Statistics:")
print(f"Mean Price: ${df_clean['Price'].mean():.2f}")
print(f"Median Price: ${df_clean['Price'].median():.2f}")
print(f"Min Price: ${df_clean['Price'].min():.2f}")
print(f"Max Price: ${df_clean['Price'].max():.2f}")
print(f"Std Dev: ${df_clean['Price'].std():.2f}")

In [ ]:
# Visualize price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['Price'], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[0].set_title('Price Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', alpha=0.3)

axes[1].scatter(df_clean['Price'], df_clean['Weekly_Sales'], alpha=0.3, s=20)
axes[1].set_title('Price vs Weekly Sales', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Weekly Sales ($)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../output/price_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Final Dataset Overview

In [ ]:
# Display final feature list
print("\n=== Final Feature List ===")
print(f"Total features: {len(df_clean.columns)}")
print("\nAll columns:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"{i:2}. {col}")

In [ ]:
# Check final data quality
print("\n=== Final Data Quality Check ===")
print(f"Total records: {len(df_clean):,}")
print(f"Missing values per column:")
print(df_clean.isnull().sum())

In [ ]:
# Display sample of cleaned data
print("\n=== Sample of Cleaned Data ===")
df_clean.head(10)

## 11. Save Processed Data

In [ ]:
# Save full processed dataset
df_clean.to_csv('../data/walmart_processed.csv', index=False)
print("✓ Processed data saved to 'data/walmart_processed.csv'")

# Save a version without NaN values (for modeling)
df_model = df_clean.dropna()
df_model.to_csv('../data/walmart_model_ready.csv', index=False)
print(f"✓ Model-ready data saved to 'data/walmart_model_ready.csv'")
print(f"   Records after dropping NaN: {len(df_model):,}")

## Key Findings Summary

### Data Overview:
- Dataset contains 6,435 weekly sales records from 45 Walmart stores
- Time period: February 2010 to 2011
- No missing values in original dataset
- No duplicate records

### Sales Patterns:
- Significant variation in sales across stores and time
- Seasonal patterns observed (higher sales in certain months)
- Holiday weeks show different sales patterns compared to non-holiday weeks

### Feature Relationships:
- Economic indicators (CPI, Unemployment, Fuel Price) show correlations with sales
- Temperature has a moderate relationship with sales
- Store-level differences are significant

### Next Steps:
1. Build demand prediction models (Linear Regression, Random Forest, Gradient Boosting)
2. Analyze price elasticity of demand
3. Develop revenue optimization algorithms
4. Create dynamic pricing recommendations

---
*Data exploration and preprocessing completed successfully!*